# Module 6 Lab 2: Appendix C.2

### Building a Table From Scratch with CREATE TABLE and INSERT

**Name:** Mac Mckay

**Course:** CS 82A

**Repo:** https://github.com/raulmckay-blip/cs82a-portfolio

## Overview

This project builds a customers table from scratch with SQL, fills it with made-up customers, and checks that they saved.

## The book's version (MySQL)

The book's `City` table from C.2, retyped with straight quotes:

```sql
CREATE TABLE `City` (
    `ID` int(11) NOT NULL auto_increment,
    `Name` char(35) NOT NULL default '',
    `CountryCode` char(3) NOT NULL default '',
    `District` char(20) NOT NULL default '',
    `Population` int(11) NOT NULL default '0',
    PRIMARY KEY (`ID`)
);

INSERT INTO City VALUES ('', 'New York', 'USA', 'New York', '10000000');
```

## Translating it to SQLite

- `create database` + `use` becomes `sqlite3.connect('lab2.db')`
- `int(11)` becomes `INTEGER`
- `char(35)` becomes `TEXT`
- `auto_increment` + `PRIMARY KEY (ID)` becomes `INTEGER PRIMARY KEY`
- `''` for the ID becomes `NULL`
- `describe City` becomes `PRAGMA table_info(...)`

Plus `REAL` for `total_spent`, since it has cents.

## Connect to a new database

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect('lab2.db')

New file so I don't overwrite Lab 1's tables.

## CREATE TABLE

Same idea as the book: `NOT NULL` with defaults, and the id numbers itself.

In [ ]:
# drop it first so re-running starts clean
con.execute("DROP TABLE IF EXISTS customers")

con.execute('''
    CREATE TABLE customers (
        id          INTEGER PRIMARY KEY,
        name        TEXT    NOT NULL DEFAULT '',
        city        TEXT    NOT NULL DEFAULT '',
        signup_year INTEGER NOT NULL DEFAULT 0,
        total_spent REAL    NOT NULL DEFAULT 0
    )
''')

pd.read_sql("SELECT name FROM sqlite_master", con)   # SQLite's version of "show tables"

## INSERT the book's way

The book uses `''` for the ID. SQLite doesn't like that:

In [ ]:
try:
    con.execute("INSERT INTO customers VALUES ('', 'Ousmane Dembele', 'Culver City', 2021, 412.50)")
except sqlite3.IntegrityError as e:
    print('SQLite error:', e)

SQLite wants `NULL` instead:

In [ ]:
con.execute("INSERT INTO customers VALUES (NULL, 'Ousmane Dembele', 'Culver City', 2021, 412.50)")

pd.read_sql("SELECT * FROM customers", con)

Dembele got id 1 automatically.

## INSERT the rest

In [ ]:
# listing the columns so I can skip id
con.execute('''
    INSERT INTO customers (name, city, signup_year, total_spent)
    VALUES
        ('Rodri',             'Santa Monica', 2023, 88.25),
        ('Lionel Messi',      'Venice',       2022, 1290.00),
        ('Karim Benzema',     'Inglewood',    2024, 57.80),
        ('Luka Modric',       'Los Angeles',  2020, 760.40),
        ('Cristiano Ronaldo', 'Torrance',     2025, 15.99),
        ('Kaka',              'Culver City',  2022, 534.10),
        ('Fabio Cannavaro',   'Los Angeles',  2026, 199.95),
        ('Ronaldinho',        'Santa Monica', 2023, 0.00),
        ('Andriy Shevchenko', 'Venice',       2021, 342.75)
''')

con.commit()   # without this the rows vanish when the connection closes

The names are the last 10 different Ballon d'Or winners, but the cities and numbers are made up. Listing the columns lets me skip `id`. Ronaldinho's \$0 is on purpose: he signed up but never bought anything.

## SELECT * to check the rows

In [ ]:
pd.read_sql("SELECT * FROM customers", con)

All 10 rows, numbered 1 to 10.

## Checking the structure (the book's `describe`)

In [ ]:
pd.read_sql("PRAGMA table_info(customers)", con)[['name', 'type', 'notnull', 'dflt_value', 'pk']]

Types and defaults match, and `id` is the primary key.

## Did the commit work?

Close, reopen, count:

In [ ]:
con.close()

con = sqlite3.connect('lab2.db')
pd.read_sql("SELECT COUNT(*) AS n_rows FROM customers", con)

Still 10. The rows saved.

## A quick query

Customers who spent over \$300, highest first:

In [ ]:
pd.read_sql('''
    SELECT name, city, total_spent
    FROM customers
    WHERE total_spent > 300
    ORDER BY total_spent DESC
''', con)

In [ ]:
con.close()

## AI Use

I used AI to help with syntax and refine wording on this lab.